In [2]:
import numpy as np

# ==========================================================
# 1. DEFINE STRUCTURE
# ==========================================================

# Unit conversions
ft_to_m  = 0.3048
in_to_m  = 0.0254

# Frame geometry
LENGTH = 24 * ft_to_m   # 24 ft → meters (horizontal span)
HEIGHT = 12 * ft_to_m   # 12 ft → meters (vertical height)

# Cross-sectional areas
# Vertical columns: circular, diameter = 10 inches
d_col   = 10 * in_to_m
A_col   = np.pi / 4 * d_col**2

# X-braces: rectangular 4 in × 1 in
A_brace = (4 * in_to_m) * (1 * in_to_m)

# Material property (structural steel)
E = 200e9   # Young's Modulus (Pa)

# Node coordinates (x, y) in meters
#
#   Node 2 -------- Node 3
#     |    \      /   |
#     |      Node 4   |   ← center node (X intersection)
#     |    /      \   |
#   Node 0 -------- Node 1
#
nodes = np.array([
    [0,        0      ],   # Node 0 – bottom-left
    [LENGTH,   0      ],   # Node 1 – bottom-right
    [0,        HEIGHT ],   # Node 2 – top-left
    [LENGTH,   HEIGHT ],   # Node 3 – top-right
    [LENGTH/2, HEIGHT/2],  # Node 4 – center (X intersection)
], dtype=float)

elements = np.array([
    [0, 2],   # Element 0  – left column
    [1, 3],   # Element 1  – right column
    [0, 1],   # Element 2  – bottom chord
    [2, 3],   # Element 3  – top chord
    [0, 4],   # Element 4  – brace: bottom-left  → center
    [4, 3],   # Element 5  – brace: center       → top-right
    [1, 4],   # Element 6  – brace: bottom-right → center
    [4, 2],   # Element 7  – brace: center       → top-left
])

areas = np.array([
    A_col, A_col, A_col, A_col,      # columns + chords
    A_brace, A_brace, A_brace, A_brace  # X-braces
])

n_nodes = len(nodes)
n_dof   = 2 * n_nodes

# ==========================================================
# 2. ASSEMBLE GLOBAL STIFFNESS MATRIX  (load-independent)
# ==========================================================

K = np.zeros((n_dof, n_dof))

for e, (i, j) in enumerate(elements):
    xi, yi = nodes[i];  xj, yj = nodes[j]
    dx = xj - xi;       dy = yj - yi
    L  = np.hypot(dx, dy)
    c  = dx / L;        s  = dy / L
    A  = areas[e]

    k_el = (E * A / L) * np.array([
        [ c*c,  c*s, -c*c, -c*s],
        [ c*s,  s*s, -c*s, -s*s],
        [-c*c, -c*s,  c*c,  c*s],
        [-c*s, -s*s,  c*s,  s*s]
    ])
    dof_map = np.array([2*i, 2*i+1, 2*j, 2*j+1])
    K[np.ix_(dof_map, dof_map)] += k_el

# ==========================================================
# 3. BOUNDARY CONDITIONS
# ==========================================================

# Pin supports at both base nodes (Node 0 and Node 1)
fixed_dofs = [2*0, 2*0+1, 2*1, 2*1+1]
free_dofs  = np.setdiff1d(np.arange(n_dof), fixed_dofs)

K_ff = K[np.ix_(free_dofs, free_dofs)]

# ==========================================================
# 4. WIND LOAD CASES
# ==========================================================
# Both forces are already given as lateral forces (N or lbs).
# They are applied at the top chord level, split equally between
# the two top nodes (Node 2 and Node 3) to represent a uniform
# lateral load distributed across the full 24-ft top chord.
# ==========================================================

# Wind forces [units as provided by user]
wind_forces = {
    "Wind Case 1 (P1 = 51.67)": 51.67,
    "Wind Case 2 (P2 = 88.90)": 88.90,
}

member_names = [
    "Left column  ", "Right column ", "Bottom chord ",
    "Top chord    ", "Brace (BL→C) ", "Brace (C→TR) ",
    "Brace (BR→C) ", "Brace (C→TL) "
]

labels = ["Bot-Left  (0)", "Bot-Right (1)", "Top-Left  (2)",
          "Top-Right (3)", "Center    (4)"]

# ==========================================================
# 5. SOLVE + REPORT FOR EACH LOAD CASE
# ==========================================================

for case_name, F_wind in wind_forces.items():

    # --- Build force vector ---
    # Lateral wind force split equally to Node 2 (top-left) and
    # Node 3 (top-right) as horizontal (x-direction) loads
    F = np.zeros(n_dof)
    F[2*2]     = F_wind / 2   # Node 2 – ux
    F[2*3]     = F_wind / 2   # Node 3 – ux

    # --- Solve ---
    u = np.zeros(n_dof)
    u[free_dofs] = np.linalg.solve(K_ff, F[free_dofs])

    # --- Lateral deflection at top chord ---
    # Average horizontal displacement of the two top nodes
    ux_top_left  = u[2*2]
    ux_top_right = u[2*3]
    lateral_drift_mm = (ux_top_left + ux_top_right) / 2 * 1e3
    lateral_drift_in = lateral_drift_mm / 25.4

    # Drift ratio = lateral deflection / height
    drift_ratio = (lateral_drift_mm / 1e3) / HEIGHT

    # --- Print results ---
    print("=" * 60)
    print(f"  {case_name}")
    print(f"  Total lateral force applied = {F_wind:.2f} (user units)")
    print(f"  Applied as {F_wind/2:.3f} at Node 2 and {F_wind/2:.3f} at Node 3")
    print("=" * 60)

    print()
    print("  Nodal Displacements:")
    print("  " + "-" * 54)
    disp = u.reshape(n_nodes, 2)
    for k, (ux, uy) in enumerate(disp):
        print(f"    Node {k} {labels[k]:15s}:  "
              f"ux = {ux*1e3:8.5f} mm,  uy = {uy*1e3:8.5f} mm")

    print()
    print("  Element Axial Forces:")
    print("  " + "-" * 54)
    for e, (i, j) in enumerate(elements):
        xi, yi = nodes[i];  xj, yj = nodes[j]
        dx = xj - xi;  dy = yj - yi
        L  = np.hypot(dx, dy)
        c  = dx / L;   s  = dy / L
        A  = areas[e]
        u_el        = u[[2*i, 2*i+1, 2*j, 2*j+1]]
        axial_force = (E * A / L) * np.array([-c, -s, c, s]) @ u_el
        tag = "T" if axial_force >= 0 else "C"
        print(f"    Elem {e} {member_names[e]}: "
              f"{axial_force:12.4f} (user units)  ({tag})")

    print()
    print("  ── LATERAL DRIFT SUMMARY ──────────────────────────")
    print(f"    Top-Left  Node 2  ux = {ux_top_left*1e3:8.5f} mm "
          f"({ux_top_left*1e3/25.4:.5f} in)")
    print(f"    Top-Right Node 3  ux = {ux_top_right*1e3:8.5f} mm "
          f"({ux_top_right*1e3/25.4:.5f} in)")
    print(f"    Avg lateral deflection   = {lateral_drift_mm:.5f} mm "
          f"  ({lateral_drift_in:.5f} in)")
    print(f"    Frame height             = {HEIGHT*1000:.1f} mm  "
          f"({HEIGHT/ft_to_m:.0f} ft)")
    print(f"    Drift ratio (Δ/H)        = 1 / {1/drift_ratio:.0f}")
    print()

print("=" * 60)
print("  T = Tension   |   C = Compression")
print("=" * 60)

  Wind Case 1 (P1 = 51.67)
  Total lateral force applied = 51.67 (user units)
  Applied as 25.835 at Node 2 and 25.835 at Node 3

  Nodal Displacements:
  ------------------------------------------------------
    Node 0 Bot-Left  (0)  :  ux =  0.00000 mm,  uy =  0.00000 mm
    Node 1 Bot-Right (1)  :  ux =  0.00000 mm,  uy =  0.00000 mm
    Node 2 Top-Left  (2)  :  ux =  0.00051 mm,  uy =  0.00000 mm
    Node 3 Top-Right (3)  :  ux =  0.00051 mm,  uy = -0.00000 mm
    Node 4 Center    (4)  :  ux =  0.00026 mm,  uy = -0.00000 mm

  Element Axial Forces:
  ------------------------------------------------------
    Elem 0 Left column  :      12.9175 (user units)  (T)
    Elem 1 Right column :     -12.9175 (user units)  (C)
    Elem 2 Bottom chord :       0.0000 (user units)  (T)
    Elem 3 Top chord    :      -0.0000 (user units)  (C)
    Elem 4 Brace (BL→C) :      28.8844 (user units)  (T)
    Elem 5 Brace (C→TR) :      28.8844 (user units)  (T)
    Elem 6 Brace (BR→C) :     -28.8844 (u